# Local Outlier Factor (LOF) for Anomaly Detection

## What This Notebook Covers
This notebook implements **Local Outlier Factor (LOF)** — both from scratch using NumPy and using scikit-learn's `LocalOutlierFactor` class. LOF detects anomalies by comparing each point's local density to the densities of its neighbours.

## Prerequisites
- k-Nearest Neighbours (k-NN) concept
- Distance metrics (Euclidean distance)
- Basic understanding of density estimation
- Python, NumPy, pandas, matplotlib

## Dataset
**Credit Card Fraud Detection** (Kaggle)  
Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
284,807 transactions with 30 features (V1-V28 from PCA + Time + Amount). Target: `Class` (0=normal, 1=fraud, ~0.17% fraud rate).

## Credits
- Dataset by ULB Machine Learning Group
- Breunig, Kriegel, Ng, Sander (2000) — Original LOF paper (ACM SIGMOD)
- Gradientts ML Intern Program — 2026

In [ ]:
# ============================================================
# Cell 2: All Imports
# ============================================================

import numpy as np                   # Core numerical computing
import pandas as pd                  # Data manipulation and loading
import matplotlib.pyplot as plt      # Plotting and visualisation
import seaborn as sns                # Statistical visualisation
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.model_selection import train_test_split  # Splitting
from sklearn.neighbors import LocalOutlierFactor  # Sklearn LOF
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve
)

# Reproducibility
np.random.seed(42)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports successful.')

## Part 1: Theory Recap

**Local Outlier Factor (LOF)** compares the local density of a point to the densities of its k nearest neighbours:

1. **k-distance:** Distance from a point to its k-th nearest neighbour — defines the neighbourhood radius.
2. **Reachability distance:** $\text{reach-dist}_k(p, o) = \max(\text{k-dist}(o), d(p,o))$ — smooths out very short distances.
3. **Local Reachability Density (LRD):** Inverse of the average reachability distance to the k nearest neighbours — high LRD = dense neighbourhood.
4. **LOF score:** Ratio of neighbours' LRDs to the point's LRD. LOF ≈ 1 = normal; LOF >> 1 = anomaly.
5. **Key strength:** Detects *local* anomalies — points that are outliers relative to their neighbourhood, even if they look normal globally.

## Loading the Dataset

We load the Credit Card Fraud dataset and create a subsample since LOF's naïve implementation has O(n²) distance computation.

In [ ]:
# ============================================================
# Cell 4: Load the real-world dataset
# ============================================================

try:
    df = pd.read_csv('creditcard.csv')
    print('Loaded from local file.')
except FileNotFoundError:
    from sklearn.datasets import fetch_openml
    print('Local file not found. Downloading from OpenML...')
    credit = fetch_openml(data_id=1597, as_frame=True, parser='auto')
    df = credit.frame
    if 'Class' not in df.columns:
        df = df.rename(columns={df.columns[-1]: 'Class'})
    df['Class'] = df['Class'].astype(int)
    print('Downloaded successfully.')

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:\n{df["Class"].value_counts()}')
print(f'\nFraud rate: {df["Class"].mean()*100:.3f}%')

print('\n--- First 5 rows ---')
df.head()

In [ ]:
print('--- Dataset Info ---')
print(df.info())
print('\n--- Statistical Summary ---')
df.describe()

## Preprocessing

Scale `Time` and `Amount`. Create a smaller subsample for the from-scratch implementation (LOF's distance matrix is O(n²) in memory and computation).

In [ ]:
# ============================================================
# Cell 5: Preprocessing
# ============================================================

# Step 1: Scale Time and Amount
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = scaler.fit_transform(df[['Time']])
df_clean = df.drop(['Time', 'Amount'], axis=1)

# Step 2: Check for nulls
print(f'Null values: {df_clean.isnull().sum().sum()}')

# Step 3: Subsample for scratch (LOF has O(n^2) distance computation)
fraud = df_clean[df_clean['Class'] == 1]
normal = df_clean[df_clean['Class'] == 0].sample(n=2000, random_state=42)
df_sub = pd.concat([normal, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(f'Class distribution:\n{df_sub["Class"].value_counts()}')

# Step 4: Use a subset of features for scratch (manage dimensionality)
feature_cols = ['V1', 'V2', 'V3', 'V4', 'V10', 'V11', 'V14', 'V17', 'scaled_Amount']
X_sub = df_sub[feature_cols].values
y_sub = df_sub['Class'].values

# Scale all selected features
scaler_sub = StandardScaler()
X_sub = scaler_sub.fit_transform(X_sub)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)

print(f'\nTraining set: {X_train.shape}')
print(f'Test set: {X_test.shape} (fraud: {y_test.sum()}, normal: {(y_test==0).sum()})')

## Part 2: From Scratch Implementation

We implement LOF from scratch using only NumPy.

**What we are building:** A `LOFScratch` class that:
1. Computes pairwise Euclidean distances
2. Finds k nearest neighbours for each point
3. Computes reachability distances, LRD, and LOF scores

**Why from scratch?** Understanding the 4-level nested computation (k-distance → reachability distance → LRD → LOF) is the most common interview question about LOF. Building it manually makes this chain crystal clear.

In [ ]:
# ============================================================
# Cell 7: From-Scratch LOF (NumPy only)
# ============================================================

class LOFScratch:
    """
    Local Outlier Factor (LOF) for anomaly detection.
    Computes LOF scores that compare each point's local density
    to its neighbours' densities.
    LOF ≈ 1 → normal, LOF >> 1 → anomaly.
    """
    
    def __init__(self, n_neighbors=20):
        # INTERVIEW NOTE: n_neighbors (k) is the key hyperparameter.
        # Typical range: 10-50. Smaller k = more local, noisier.
        # Larger k = more global, smoother but may miss local anomalies.
        self.n_neighbors = n_neighbors
    
    def fit_predict(self, X):
        """
        Compute LOF scores for all points in X.
        Returns (lof_scores, labels).
        """
        self.X = np.array(X, dtype=np.float64)
        self.n = len(X)
        k = min(self.n_neighbors, self.n - 1)
        
        # Step 1: Compute pairwise distance matrix
        # INTERVIEW NOTE: This is the bottleneck — O(n^2 * d).
        # Tree-based methods (KD-tree, Ball-tree) reduce this to O(n*d*log n).
        print('  Computing pairwise distances...')
        self.dist_matrix = self._compute_distance_matrix()
        
        # Step 2: Find k nearest neighbours and k-distances
        # INTERVIEW NOTE: k-distance(p) = distance to the k-th nearest neighbour.
        # This defines the "neighbourhood radius" of each point.
        print('  Finding k nearest neighbours...')
        self.k_neighbors, self.k_distances = self._find_k_neighbors(k)
        
        # Step 3: Compute reachability distances
        # INTERVIEW NOTE: reach-dist(p, o) = max(k-dist(o), d(p,o))
        # This smooths out very short distances, making LRD more stable.
        
        # Step 4: Compute Local Reachability Density (LRD)
        print('  Computing LRD...')
        self.lrd = self._compute_lrd(k)
        
        # Step 5: Compute LOF scores
        # INTERVIEW NOTE: LOF(p) = mean(LRD(o) / LRD(p)) for all neighbours o.
        # If neighbours have higher LRD than p, then p is less dense = anomaly.
        print('  Computing LOF scores...')
        self.lof_scores = self._compute_lof(k)
        
        return self.lof_scores
    
    def _compute_distance_matrix(self):
        """Compute all pairwise Euclidean distances."""
        # Using vectorised broadcasting for efficiency
        # ||x - y||^2 = ||x||^2 + ||y||^2 - 2*x·y
        X_sq = np.sum(self.X ** 2, axis=1)
        dist_sq = X_sq[:, None] + X_sq[None, :] - 2 * self.X @ self.X.T
        # Clamp to avoid negative values from floating-point errors
        dist_sq = np.maximum(dist_sq, 0)
        return np.sqrt(dist_sq)
    
    def _find_k_neighbors(self, k):
        """Find k nearest neighbours and k-distances for each point."""
        k_neighbors = []
        k_distances = np.zeros(self.n)
        
        for i in range(self.n):
            # Sort distances (exclude self at index 0)
            sorted_indices = np.argsort(self.dist_matrix[i])[1:k+1]
            k_neighbors.append(sorted_indices)
            # k-distance = distance to the k-th nearest neighbour
            k_distances[i] = self.dist_matrix[i, sorted_indices[-1]]
        
        return k_neighbors, k_distances
    
    def _compute_lrd(self, k):
        """
        Compute Local Reachability Density for each point.
        LRD(p) = 1 / (average reachability distance of p's k neighbours)
        """
        lrd = np.zeros(self.n)
        
        for i in range(self.n):
            # Compute reachability distances to all k neighbours
            reach_dists = np.zeros(k)
            for j_idx, j in enumerate(self.k_neighbors[i]):
                # INTERVIEW NOTE: This max() operation is the key insight.
                # It prevents very close points from having extreme LRD values.
                reach_dists[j_idx] = max(self.k_distances[j], self.dist_matrix[i, j])
            
            # LRD = inverse of mean reachability distance
            mean_reach_dist = np.mean(reach_dists)
            lrd[i] = 1.0 / (mean_reach_dist + 1e-10)  # Epsilon to avoid division by zero
        
        return lrd
    
    def _compute_lof(self, k):
        """
        Compute Local Outlier Factor for each point.
        LOF(p) = mean(LRD(neighbour) / LRD(p)) for all k neighbours
        """
        lof_scores = np.zeros(self.n)
        
        for i in range(self.n):
            # INTERVIEW NOTE: LOF compares LOCAL densities.
            # If all neighbours have similar density → LOF ≈ 1 (normal).
            # If neighbours are much denser → LOF >> 1 (anomaly).
            neighbor_lrd_sum = sum(self.lrd[j] for j in self.k_neighbors[i])
            lof_scores[i] = (neighbor_lrd_sum / k) / (self.lrd[i] + 1e-10)
        
        return lof_scores


print('LOFScratch class defined successfully.')

## Fitting and Evaluating the From-Scratch LOF

We apply LOF to the combined training+test data (LOF is transductive — it scores the data it's fitted on). For evaluation, we use the known labels.

In [ ]:
# ============================================================
# Cell 8: Fit scratch model and evaluate
# ============================================================

# LOF is transductive: we score the entire dataset at once
# Combine train + test for LOF scoring
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

print(f'Running LOF from scratch on {len(X_all)} points with k=20...')
lof_scratch = LOFScratch(n_neighbors=20)
scratch_lof_scores = lof_scratch.fit_predict(X_all)

# Extract test set scores (they are at the end of the combined array)
n_train = len(X_train)
test_lof_scores = scratch_lof_scores[n_train:]

# Higher LOF = more anomalous
contamination = y_test.mean()
threshold = np.percentile(test_lof_scores, (1 - contamination) * 100)
scratch_labels = np.where(test_lof_scores > threshold, 1, 0)

print('\n--- From-Scratch LOF Results ---')
print(classification_report(y_test, scratch_labels, target_names=['Normal', 'Fraud']))

# AUC-ROC (higher LOF = more anomalous = higher fraud probability)
scratch_auc = roc_auc_score(y_test, test_lof_scores)
print(f'AUC-ROC: {scratch_auc:.4f}')

scratch_ap = average_precision_score(y_test, test_lof_scores)
print(f'Average Precision (PR-AUC): {scratch_ap:.4f}')

## Part 3: Sklearn Implementation

Scikit-learn's `LocalOutlierFactor` provides:
- **Ball-tree/KD-tree** acceleration for neighbour queries (O(n·log n) instead of O(n²))
- `negative_outlier_factor_` attribute (more negative = more anomalous — note the sign convention)
- `novelty=True` mode for scoring new unseen data

Key difference from scratch: sklearn uses efficient tree data structures and returns *negative* LOF scores by convention.

In [ ]:
# ============================================================
# Cell 10: Sklearn LOF Implementation
# ============================================================

# Fit sklearn LOF (transductive mode)
sklearn_lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=float(contamination),
    novelty=False  # Transductive: scores training data
)

# fit_predict returns labels directly (-1 = outlier, 1 = inlier)
sklearn_all_preds = sklearn_lof.fit_predict(X_all)
sklearn_all_scores = sklearn_lof.negative_outlier_factor_  # More negative = more anomalous

# Extract test set results
sklearn_test_preds = sklearn_all_preds[n_train:]
sklearn_test_scores = sklearn_all_scores[n_train:]
sklearn_labels = np.where(sklearn_test_preds == -1, 1, 0)

print('--- Sklearn LOF Results ---')
print(classification_report(y_test, sklearn_labels, target_names=['Normal', 'Fraud']))

# AUC (negate sklearn scores: more negative = more anomalous → positive for AUC)
sklearn_auc = roc_auc_score(y_test, -sklearn_test_scores)
print(f'AUC-ROC: {sklearn_auc:.4f}')

sklearn_ap = average_precision_score(y_test, -sklearn_test_scores)
print(f'Average Precision (PR-AUC): {sklearn_ap:.4f}')

# --- Direct Comparison ---
print('\n=== Scratch vs Sklearn Comparison ===')
print(f'{"Metric":<25} {"Scratch":<15} {"Sklearn":<15}')
print(f'{"AUC-ROC":<25} {scratch_auc:<15.4f} {sklearn_auc:<15.4f}')
print(f'{"Avg Precision (PR-AUC)":<25} {scratch_ap:<15.4f} {sklearn_ap:<15.4f}')

## Visualisations

1. **LOF Score Distribution** — comparing normal vs fraud LOF scores
2. **ROC Curve** — scratch vs sklearn comparison

In [ ]:
# ============================================================
# Cell 11: Visualisations
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: LOF Score Distribution ---
ax1 = axes[0]
# Clip LOF scores for better visualisation (some outliers have very high LOF)
clipped_scores = np.clip(test_lof_scores, 0, 5)
ax1.hist(clipped_scores[y_test == 0], bins=50, alpha=0.7, label='Normal', color='steelblue', density=True)
ax1.hist(clipped_scores[y_test == 1], bins=50, alpha=0.7, label='Fraud', color='crimson', density=True)
ax1.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5, label='LOF = 1 (baseline)')
ax1.set_xlabel('LOF Score (clipped at 5)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('LOF Score Distribution: Normal vs Fraud', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# --- Plot 2: ROC Curve ---
ax2 = axes[1]
fpr_s, tpr_s, _ = roc_curve(y_test, test_lof_scores)  # Higher LOF = anomaly
fpr_sk, tpr_sk, _ = roc_curve(y_test, -sklearn_test_scores)  # More negative = anomaly
ax2.plot(fpr_s, tpr_s, label=f'Scratch (AUC={scratch_auc:.3f})', color='darkorange', linewidth=2)
ax2.plot(fpr_sk, tpr_sk, label=f'Sklearn (AUC={sklearn_auc:.3f})', color='steelblue', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve: Scratch vs Sklearn LOF', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

# --- Plot 3: Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))
prec, rec, _ = precision_recall_curve(y_test, -sklearn_test_scores)
ax.plot(rec, prec, color='steelblue', linewidth=2, label=f'LOF (AP={sklearn_ap:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve (LOF Anomaly Detection)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

Key hyperparameters for LOF:

1. **`n_neighbors` (k)** — controls locality. Too small = noisy, too large = loses local sensitivity.
2. **`metric`** — distance function. Euclidean is standard, but Manhattan or Minkowski may work better for certain data.

We vary both and measure AUC-ROC.

In [ ]:
# ============================================================
# Cell 13: Hyperparameter Experiments
# ============================================================

# --- Experiment 1: n_neighbors effect ---
k_values = [5, 10, 15, 20, 30, 50, 75, 100]
k_aucs = []

for k in k_values:
    lof_exp = LocalOutlierFactor(n_neighbors=k, contamination=float(contamination), novelty=False)
    lof_exp.fit_predict(X_all)
    scores = lof_exp.negative_outlier_factor_[n_train:]
    auc_val = roc_auc_score(y_test, -scores)
    k_aucs.append(auc_val)
    print(f'n_neighbors={k:<4d}  AUC-ROC={auc_val:.4f}')

# --- Experiment 2: Distance metric effect ---
metrics = ['euclidean', 'manhattan', 'chebyshev', 'minkowski']
metric_aucs = []
best_k = k_values[np.argmax(k_aucs)]

for metric in metrics:
    lof_exp = LocalOutlierFactor(n_neighbors=best_k, contamination=float(contamination),
                                  metric=metric, novelty=False)
    lof_exp.fit_predict(X_all)
    scores = lof_exp.negative_outlier_factor_[n_train:]
    auc_val = roc_auc_score(y_test, -scores)
    metric_aucs.append(auc_val)
    print(f'metric={metric:<15}  AUC-ROC={auc_val:.4f}')

# --- Plot results ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(k_values, k_aucs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(x=best_k, color='crimson', linestyle='--', label=f'Best k={best_k}')
axes[0].set_xlabel('Number of Neighbours (k)', fontsize=12)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('Effect of n_neighbors on LOF Performance', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

bars = axes[1].bar(metrics, metric_aucs, color=['steelblue', 'coral', 'mediumseagreen', 'mediumpurple'])
axes[1].set_xlabel('Distance Metric', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title(f'Effect of Distance Metric (k={best_k})', fontsize=13, fontweight='bold')
for bar, val in zip(bars, metric_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Key Conceptual Question: *"Why is LOF better than global threshold methods for anomaly detection in datasets with varying cluster densities?"*

**Answer Structure:**

1. **The problem with global methods:** A global density threshold (like KDE with a single cutoff) fails when the data has clusters of different densities. A point at the edge of a sparse cluster might have the same global density as a point deep inside a dense cluster — but only the former is anomalous.

2. **LOF's local comparison:** LOF compares each point's density to its *neighbours'* densities. A sparse-cluster edge point has LOF >> 1 because its neighbours (in the sparse cluster) still have higher density. A point at the edge of a dense cluster has LOF ≈ 1 because its dense neighbours balance out.

3. **Concrete example:** Consider two clusters — Cluster A (tight, 100 points) and Cluster B (loose, 50 points). A global threshold would flag many Cluster B points as anomalous. LOF correctly identifies that Cluster B points are normal *relative to their neighbourhood*.

4. **The reachability distance smoothing:** The max(k-dist(o), d(p,o)) formula prevents very close points from having infinite LRD, stabilising the LOF computation in dense regions.

5. **Trade-off:** LOF's locality is also its limitation — it's O(n²) naïvely and struggles in high dimensions where distances converge.

## Key Takeaways

1. **LOF is a *local* anomaly detector** — it compares each point's density to its neighbours, making it uniquely effective for datasets with clusters of different densities where global methods fail.

2. **The 4-level computation chain** (k-distance → reachability distance → LRD → LOF) is the most common interview topic. Each level addresses a specific statistical challenge: defining neighbourhood, smoothing close distances, estimating density, and comparing densities.

3. **LOF ≈ 1 means normal; LOF >> 1 means anomaly.** This is because the ratio compares neighbours' LRDs (potentially high) to the point's own LRD (low for outliers).

4. **k (n_neighbors) controls the definition of "local"** — too small makes it sensitive to noise; too large makes it behave like a global method. The original paper recommends testing k from MinPts to MaxPts and taking the maximum LOF.

5. **LOF is O(n²·d) naïvely** — the distance matrix is the bottleneck. For large datasets, sklearn uses Ball-tree/KD-tree to bring this down to O(n·d·log n). For truly massive data, consider Isolation Forest instead.